### AI Operations (AIOps) — Question 2

## Step 0 — Setup
Install dependencies (skip if already installed) and import libraries.

In [ ]:
# !pip install mlflow scikit-learn pandas --quiet

import mlflow
import mlflow.sklearn
import pandas as pd
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score

mlflow.set_tracking_uri("http://localhost:5000")
mlflow.set_experiment("mnist-mlp")
print("Tracking URI:", mlflow.get_tracking_uri())

## Step 1 — The starter script (un-instrumented)
This is the "before" version — plain scikit-learn, no tracking at all. Run it once just to confirm it works.

In [ ]:
X, y = fetch_openml("mnist_784", version=1, return_X_y=True, as_frame=False)
X = X.astype("float32") / 255.0
y = y.astype("int64")
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42,stratify=y)

def train_and_evaluate(learning_rate=0.001, batch_size=32):
    model = MLPClassifier(
        hidden_layer_sizes=(128,),
        learning_rate_init=learning_rate,
        batch_size=batch_size,
        max_iter=20,
        random_state=42,
        early_stopping=True,
        validation_fraction=0.1
    )

    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    test_accuracy = accuracy_score(y_test, preds)
    return model, test_accuracy


## Step 2 & 3 — Instrument it: manual logging
Wrap training in `with mlflow.start_run():` and log parameters, metrics, and a tag.

In [ ]:
def train_and_log(learning_rate=0.001, batch_size=32, run_name=None):
    with mlflow.start_run(run_name=run_name):
        # --- parameters (at least 3) ---
        mlflow.log_param("learning_rate", learning_rate)
        mlflow.log_param("batch_size", batch_size)
        mlflow.log_param("hidden_layer_sizes", "(128,)")

        model, test_accuracy = train_and_evaluate(learning_rate, batch_size)

        mlflow.log_metric("test_accuracy", test_accuracy)


        # Log training/validation curves
        for epoch, loss in enumerate(model.loss_curve_):
            mlflow.log_metric("train_loss", loss, step=epoch)

        for epoch, val_acc in enumerate(model.validation_scores_):
            mlflow.log_metric("val_accuracy", val_acc, step=epoch)

        mlflow.set_tag("team", "data-science")
        mlflow.sklearn.log_model(model, name="model")

        run_id = mlflow.active_run().info.run_id
        print(
            f"Run {run_id} | "
            f"lr={learning_rate} | "
            f"batch={batch_size} | "
            f"accuracy={test_accuracy:.4f}")
        
        return run_id



## Step 4 — Sweep: 4 runs varying `learning_rate`,`batch_size`
Re-run with different values and log each as its own MLflow run.

In [ ]:
runs = []

configs = [
    (0.001, 32),
    (0.001, 64),
    (0.001, 128),
    (0.01, 32),
    (0.01, 64),
    (0.01, 128)
]

for lr, batch in configs:

    run_id = train_and_log(
        learning_rate=lr,
        batch_size=batch,
        run_name=f"mlp-lr{lr}-batch{batch}"
    )

    runs.append(run_id)

## Step 6 — Find the best run with `mlflow.search_runs()`
No need to open the UI to find the winner — query it directly.

In [ ]:
runs_df = mlflow.search_runs(
    experiment_names=["mnist-mlp"],
    order_by=["metrics.accuracy DESC"],
)
display_cols = [
    "run_id", "tags.mlflow.runName", "params.learning_rate", "params.batch_size", "metrics.accuracy","metrics.train_loss","metrics.val_accuracy"]
print(runs_df[display_cols].head(10).to_string(index=False))


## Step 7 — Open the MLflow UI

1. Go to **http://localhost:5000** in your browser.
2. Open the **mnist-mlp** experiment.
3. Select all **6 runs** from this notebook and click **Compare**.
4. Compare the runs using learning rate, batch size, accuracy, train loss, and validation accuracy.
---
### Deliverable checklist

- [ ] Screenshot of the 6-run comparison view in the MLflow UI
- [ ] Identify the best-performing run
- [ ] Analyze train_loss vs. val_accuracy for overfitting
- [ ] Determine whether learning rate or batch size has the larger effect
- [ ] Submit the exact mlflow.log_param / mlflow.log_metric code